In [ ]:
# 1. IMPORTATION DES LIBRAIRIES
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LassoCV, RidgeCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder, StandardScaler, QuantileTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.base import clone
from scipy.optimize import minimize


In [ ]:
# 1. CHARGEMENT ET CONCATÉNATION

# Chargement des datasets d'entraînement et de test
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

# Outliers à enlever selon la documentation de De Cock (GrLivArea > 4000)
train_df = train_df[train_df["GrLivArea"] < 4000].reset_index(drop=True)

ntrain = len(train_df)
y_train_log = np.log1p(train_df["SalePrice"].copy())

all_data = pd.concat([train_df.drop(columns=["SalePrice"]), test_df], axis=0).reset_index(drop=True)

# Sauvegarde des IDs pour la soumission finale Kaggle, puis suppression
test_ids = test_df['Id'].copy()
all_data = all_data.drop(['Id'], axis=1)

print(f"Shape initiale all_data : {all_data.shape}")


In [ ]:
missing_counts = all_data.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

print(f"\nDONNÉES MANQUANTES ({len(missing_counts)} colonnes):")
if len(missing_counts) > 0:
    for col, count in missing_counts.head(100).items():
        pct = 100 * count / len(all_data)
        print(f"   {col:20s} : {count:4d} ({pct:5.1f}%)")
else:
    print("   Aucune")


In [ ]:
# Suppression des lignes contenant des NA considérés comme des erreurs dans train.csv
# Renseigne ici UNIQUEMENT les colonnes où un NA est une vraie erreur de saisie.
error_missing_cols = [
    'MSZoning',
    'BsmtFullBath', 
    'Functional',
    'BsmtHalfBath',
    'Utilities',
    'BsmtFinSF1',
    'Exterior2nd',
    'Exterior1st',
    'Electrical',
    'TotalBsmtSF',
    'BsmtUnfSF',
    'BsmtFinSF2',
    'KitchenQual',
    'GarageArea',
    'GarageCars',
    'SaleType'
]

print(f"Colonnes à vérifier pour NA erreurs : {error_missing_cols}")

if len(error_missing_cols) > 0:
    valid_error_cols = [c for c in error_missing_cols if c in train_df.columns]
    dropped_error_cols = [c for c in error_missing_cols if c not in train_df.columns]

    if len(dropped_error_cols) > 0:
        print(f"Colonnes ignorées (absentes de train.csv): {dropped_error_cols}")

    n_before_drop = len(train_df)
    train_df = train_df.dropna(subset=valid_error_cols).reset_index(drop=True)
    n_removed = n_before_drop - len(train_df)
    
    # RE-ALIGNEMENT CRITIQUE DES DONNÉES CONCATÉNÉES (Correction de bug)
    ntrain = len(train_df)
    y_train_log = np.log1p(train_df["SalePrice"].copy())
    all_data = pd.concat([train_df.drop(columns=["SalePrice"]), test_df], axis=0).reset_index(drop=True)
    
    print(f"Lignes supprimées sur train_df (NA erreurs): {n_removed}")
    print(f"Nouvelle shape all_data : {all_data.shape} (ntrain={ntrain})")
else:
    print("Aucune suppression de lignes: error_missing_cols est vide.")


In [ ]:
# 2. NETTOYAGE GLOBAL SUR ALL_DATA

def apply_base_preprocessing(data):
    """Applique le nettoyage de base et le Feature Engineering sur l'ensemble complet (Train + Test)"""
    df = data.copy()

    # 1. NA signifiant "Absence de l'équipement" -> Catégorie 'None'
    cols_none = ['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
                 'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish',
                 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType']
    for col in cols_none:
        if col in df.columns:
            df[col] = df[col].fillna('None')

    # 2. NA signifiant 0 pour les variables numériques
    cols_zero = ['GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 
                 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']
    for col in cols_zero:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # 3. Imputation par la médiane globale pour la façade (LotFrontage)
    if 'LotFrontage' in df.columns:
        df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

    # 4. Conversion des variables numériques qui sont en réalité catégorielles (MSSubClass, YrSold, MoSold)
    df['MSSubClass'] = df['MSSubClass'].astype(str)
    df['YrSold'] = df['YrSold'].astype(str)
    df['MoSold'] = df['MoSold'].astype(str)

    # 5. Feature Engineering
    # Surface totale habitable (RDC + 1er + 2eme + sous-sol)
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    
    # Nombre total de salles de bain (les demi-salles comptent pour 0.5)
    df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
    
    # Flag binaire indiquant si la maison a été rénovée
    df['IsRemodeled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)
    
    # Calcul des âges par rapport à l'année de vente
    df['AgeBuilt'] = df['YrSold'].astype(int) - df['YearBuilt']
    df['AgeRemodAdd'] = df['YrSold'].astype(int) - df['YearRemodAdd']
    
    # Age du garage (si pas de garage, on prend l'âge de construction de la maison)
    df['AgeGarage'] = df['YrSold'].astype(int) - df['GarageYrBlt']
    df.loc[df['GarageYrBlt'] == 0, 'AgeGarage'] = df.loc[df['GarageYrBlt'] == 0, 'AgeBuilt']
    
    # Surface de porch total
    df['TotalPorchSF'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch'] + df['WoodDeckSF']

    # Interactions qualitatives utiles pour les modèles linéaires
    df['QualSF'] = df['OverallQual'] * df['TotalSF']

    # 6. Traitement de la Multicolinéarité pour les modèles linéaires régularisés
    # On retire les colonnes redondantes pour éviter le surapprentissage
    cols_to_drop = ['TotRmsAbvGrd', 'GarageYrBlt']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    return df

# Application du nettoyage
all_data_clean = apply_base_preprocessing(all_data)

# 3. SÉPARATION TRAIN / TEST
X_train_clean = all_data_clean.iloc[:ntrain].copy()
X_test_clean = all_data_clean.iloc[ntrain:].copy()

print(f"Shape X_train_clean : {X_train_clean.shape}")
print(f"Shape X_test_clean  : {X_test_clean.shape}")


In [ ]:
# 4. ENCODAGE AVANCÉ

class AdvancedCategoricalEngineer(BaseEstimator, TransformerMixin):
    """Transformateur Scikit-Learn pour encodage ordinal et Target Encoding (sklearn)."""
    def __init__(self, smoothing=10.0):
        self.smoothing = smoothing
        self.qual_mapping = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
        self.ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                             'HeatingQC', 'KitchenQual', 'FireplaceQu',
                             'GarageQual', 'GarageCond', 'PoolQC']
        self.te = None
        self.nominal_cols = None

    def fit(self, X, y):
        X_copy = X.copy()
        for col in self.ordinal_cols:
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(self.qual_mapping).fillna(0)

        self.nominal_cols = X_copy.select_dtypes(include=['object', 'string']).columns.tolist()

        for col in self.nominal_cols:
            X_copy[col] = X_copy[col].astype(str).fillna('Missing')

        self.te = TargetEncoder(
            categories='auto',
            target_type='continuous',
            smooth=self.smoothing,
            cv=5,
            shuffle=True,
            random_state=42
        )
        self.te.fit(X_copy[self.nominal_cols], y)
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in self.ordinal_cols:
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(self.qual_mapping).fillna(0)

        if self.nominal_cols:
            for col in self.nominal_cols:
                if col in X_copy.columns:
                    X_copy[col] = X_copy[col].astype(str).fillna('Missing')
            X_copy[self.nominal_cols] = self.te.transform(X_copy[self.nominal_cols])

        return X_copy


In [ ]:
# 5. VALIDATION CROISÉE - RÉGRESSION LASSO BASELINE

kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []

print("\nDébut Validation Croisée Lasso (5-Fold)...")
pipeline_m1_lasso = Pipeline([
    ('encoder', AdvancedCategoricalEngineer(smoothing=10.0)),
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=0.0005, max_iter=20000))
])

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_clean)):
    X_tr, y_tr = X_train_clean.iloc[train_idx], y_train_log.iloc[train_idx]
    X_val, y_val = X_train_clean.iloc[val_idx], y_train_log.iloc[val_idx]

    pipeline_m1_lasso.fit(X_tr, y_tr)
    preds = pipeline_m1_lasso.predict(X_val)

    fold_rmse = np.sqrt(mean_squared_error(y_val, preds))
    rmse_scores.append(fold_rmse)
    print(f"Fold {fold + 1} | RMSE: {fold_rmse:.5f}")

print(f"\nRMSE Moyen (CV) : {np.mean(rmse_scores):.5f} (+/- {np.std(rmse_scores):.5f})")


In [ ]:
# 6. ANALYSE MULTICOLINÉARITÉ
print("\n" + "="*70)
print("ANALYSE MULTICOLINÉARITÉ")
print("="*70)

numeric_features = X_train_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_clean.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
correlation_matrix = X_train_clean[numeric_features].corr()

print("\nCorrélations fortes (|r| > 0.85):")
print("-" * 60)

high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.85:
            col_i = correlation_matrix.columns[i]
            col_j = correlation_matrix.columns[j]
            corr_value = correlation_matrix.iloc[i, j]
            high_corr_pairs.append((col_i, col_j, corr_value))
            print(f"   {col_i:20s} <-> {col_j:20s} : {corr_value:.3f}")

if len(high_corr_pairs) == 0:
    print("   Aucune corrélation forte détectée")

# Heatmap
print("\nHeatmap de corrélation (variables numériques):")
plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, annot=False, square=True, cbar_kws={'label': 'Corrélation'})
plt.title('Matrice de corrélation - Variables numériques')
plt.tight_layout()
plt.show()


# PHASE 3 : MODÉLISATION (BASELINE → OPTIMISATION)

In [ ]:
# 9. PIPELINES OPTIMISÉS (QuantileTransformer + Fine Alphas + multi-modèles linéaires)
print("\n" + "="*70)
print("PIPELINES OPTIMISÉS POUR LES MODÈLES LINÉAIRES (Lasso, Ridge, ElasticNet)")
print("="*70)

numeric_transformer_optimized = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', QuantileTransformer(output_distribution='normal', random_state=42))
])

categorical_transformer_optimized = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_encoder', TargetEncoder(categories='auto', target_type='continuous', smooth='auto', cv=5))
])

numeric_features_valid = [c for c in numeric_features if c in X_train_clean.columns]
categorical_features_valid = [c for c in categorical_features if c in X_train_clean.columns]

preprocessor_optimized = ColumnTransformer(transformers=[
    ('num', numeric_transformer_optimized, numeric_features_valid),
    ('cat', categorical_transformer_optimized, categorical_features_valid)
])

# 1. LassoCV
lasso_optimized = LassoCV(alphas=np.logspace(-6, 1, 1000), cv=5, random_state=42, max_iter=20000, tol=1e-4)
pipeline_lasso = Pipeline(steps=[
    ('preprocessor', preprocessor_optimized),
    ('model', lasso_optimized)
])

# 2. RidgeCV
ridge_optimized = RidgeCV(alphas=np.logspace(-3, 3, 200), cv=5)
pipeline_ridge = Pipeline(steps=[
    ('preprocessor', preprocessor_optimized),
    ('model', ridge_optimized)
])

# 3. ElasticNetCV
enet_optimized = ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1], alphas=np.logspace(-6, 1, 200), cv=5, random_state=42, max_iter=20000)
pipeline_enet = Pipeline(steps=[
    ('preprocessor', preprocessor_optimized),
    ('model', enet_optimized)
])

print("\nEntraînement des modèles linéaires optimisés...")
y_train_for_fit = y_train_log.iloc[: X_train_clean.shape[0] ].reset_index(drop=True)

pipeline_lasso.fit(X_train_clean, y_train_for_fit)
pipeline_ridge.fit(X_train_clean, y_train_for_fit)
pipeline_enet.fit(X_train_clean, y_train_for_fit)

print(f"   Lasso alpha optimal       : {pipeline_lasso.named_steps['model'].alpha_:.6f}")
print(f"   Ridge alpha optimal       : {pipeline_ridge.named_steps['model'].alpha_:.6f}")
print(f"   ElasticNet alpha optimal  : {pipeline_enet.named_steps['model'].alpha_:.6f}")
print(f"   ElasticNet l1_ratio opt   : {pipeline_enet.named_steps['model'].l1_ratio_:.2f}")


In [ ]:
# 9B. ENSEMBLE AVANCÉ AVEC POIDS OPTIMISÉS PAR SCIPY
print("\n" + "="*70)
print("ENSEMBLE OPTIMISÉ : LINEAR MODELS + BOOSTING (HGB & XGBoost) + RANDOM FOREST")
print("="*70)

ensemble_models = {
    "lasso": pipeline_lasso,
    "ridge": pipeline_ridge,
    "enet": pipeline_enet,
    "hgb": Pipeline(steps=[
        ('preprocessor', preprocessor_optimized),
        ('model', HistGradientBoostingRegressor(
            loss='squared_error',
            learning_rate=0.03,
            max_iter=450,
            max_depth=5,
            l2_regularization=0.1,
            random_state=42
        ))
    ]),
    "xgb": Pipeline(steps=[
        ('preprocessor', preprocessor_optimized),
        ('model', XGBRegressor(
            n_estimators=700,
            learning_rate=0.02,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "rf": Pipeline(steps=[
        ('preprocessor', preprocessor_optimized),
        ('model', RandomForestRegressor(
            n_estimators=500,
            max_depth=15,
            min_samples_leaf=2,
            max_features='sqrt',
            random_state=42,
            n_jobs=-1
        ))
    ])
}

kf_ensemble = KFold(n_splits=5, shuffle=True, random_state=42)
oof_predictions = {}
test_predictions = {}
cv_rmse = {}

for name, model_template in ensemble_models.items():
    print(f"\nModèle: {name}")
    oof = np.zeros(X_train_clean.shape[0])
    fold_test_predictions = []

    for fold, (train_idx, val_idx) in enumerate(kf_ensemble.split(X_train_clean), 1):
        model_fold = clone(model_template)
        X_tr = X_train_clean.iloc[train_idx]
        X_val = X_train_clean.iloc[val_idx]
        y_tr = y_train_for_fit.iloc[train_idx]
        y_val = y_train_for_fit.iloc[val_idx]

        model_fold.fit(X_tr, y_tr)
        val_pred = model_fold.predict(X_val)
        test_pred = model_fold.predict(X_test_clean)

        oof[val_idx] = val_pred
        fold_test_predictions.append(test_pred)

        fold_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
        print(f"   Fold {fold} | RMSE: {fold_rmse:.5f}")

    oof_predictions[name] = oof
    test_predictions[name] = np.mean(fold_test_predictions, axis=0)
    cv_rmse[name] = np.sqrt(mean_squared_error(y_train_for_fit, oof))
    print(f"   RMSE CV moyen: {cv_rmse[name]:.5f}")

# OPTIMISATION DES POIDS DE L'ENSEMBLE AVEC SCIPY MINIMIZE
def objective(weights):
    weights = np.array(weights)
    weights = weights / np.sum(weights)  # Normalisation pour que la somme = 1
    blend = np.zeros_like(y_train_for_fit, dtype=float)
    for w, name in zip(weights, ensemble_models.keys()):
        blend += w * oof_predictions[name]
    return np.sqrt(mean_squared_error(y_train_for_fit, blend))

# Poids initiaux égaux
init_weights = [1.0 / len(ensemble_models)] * len(ensemble_models)
bounds = [(0, 1)] * len(ensemble_models)
constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - sum(w)})

res = minimize(objective, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
best_weights = res.x
best_weights = best_weights / np.sum(best_weights)  # Normalisation finale stricte

print("\nPoids optimisés de l'ensemble (Scipy) :")
ensemble_weights = {}
for w, name in zip(best_weights, ensemble_models.keys()):
    ensemble_weights[name] = w
    print(f"   {name:6s}: {w:.4f}  (RMSE CV individuel: {cv_rmse[name]:.5f})")

oof_ensemble = np.zeros(X_train_clean.shape[0])
y_pred_log_ensemble = np.zeros(X_test_clean.shape[0])
for name in ensemble_models:
    oof_ensemble += ensemble_weights[name] * oof_predictions[name]
    y_pred_log_ensemble += ensemble_weights[name] * test_predictions[name]

y_pred_dollars_ensemble = np.expm1(y_pred_log_ensemble)
rmse_ensemble = np.sqrt(mean_squared_error(y_train_for_fit, oof_ensemble))
mae_ensemble = mean_absolute_error(y_train_for_fit, oof_ensemble)
r2_ensemble = r2_score(y_train_for_fit, oof_ensemble)

print("\nMÉTRIQUES DE L'ENSEMBLE OPTIMISÉ PAR SCIPY:")
print(f"   RMSE: {rmse_ensemble:.5f}")
print(f"   MAE: {mae_ensemble:.5f}")
print(f"   R²: {r2_ensemble:.5f}")

final_model_name = "Ensemble Optimisé (Lasso+Ridge+ENet+HGB+XGB+RF)"
final_rmse = rmse_ensemble
final_mae = mae_ensemble
final_r2 = r2_ensemble
final_y_pred_log = y_pred_log_ensemble
final_y_pred_dollars = y_pred_dollars_ensemble


# PHASE 5 : PRÉDICTIONS & SOUMISSION

In [ ]:
# 15. PRÉDICTIONS FINALES
print("\n" + "="*70)
print("PRÉDICTIONS FINALES")
print("="*70)

print(f"\nModèle Ensemble:")
print(f"   Min: ${final_y_pred_dollars.min():,.0f}")
print(f"   Max: ${final_y_pred_dollars.max():,.0f}")
print(f"   Médiane: ${np.median(final_y_pred_dollars):,.0f}")
print(f"   Moyenne: ${final_y_pred_dollars.mean():,.0f}")
print(f"   Sélection finale: {final_model_name}")
y_pred_dollars_final = final_y_pred_dollars


In [ ]:
# 16. GÉNÉRATION FICHIERS CSV & RÉSUMÉ FINAL
print("\n" + "="*70)
print("GÉNÉRATION FICHIERS SUBMISSIONS")
print("="*70)

submission_opt = pd.DataFrame({
    'Id': test_ids.values,
    'SalePrice': y_pred_dollars_final
   })
submission_opt.to_csv('M1_Lasso_V6_v5.csv', index=False)
print(f"\nM1_Lasso_V6_v5.csv généré ({len(submission_opt)} prédictions)")

# RÉSUMÉ FINAL
print(f"\n{'='*70}")
print("RÉSUMÉ FINAL - SUBMISSION V5")
print(f"{'='*70}")

summary_text = f"""

MEILLEUR MODÈLE (parmi évalués):
    • Modèle: {final_model_name}
    • RMSE (CV): {final_rmse:.5f}
    • MAE (CV): {final_mae:.5f}
    • R² (CV): {final_r2:.5f}
    • Scaler recommandé: QuantileTransformer
    • Optimisation des poids: SLSQP (Scipy minimize)

FICHIER GÉNÉRÉ:
    • M1_Lasso_V6_v5.csv

"""

print(summary_text)
